# Thematic Investing: A Risk-based Perspective 再現実装

このNotebookは、`docs/notes/review.md` に整理した文献レビューをもとに、文献の方法論へ寄せた再現実装を行う。実データが `data/raw/` に配置されていれば日本株テーマバスケット + Barra GEMLTL 形式のデータを読み込み、未配置の場合は同じスキーマの合成データで smoke test を行う。

合成データの結果は文献の実証結果ではない。APWC、release前後60カレンダー日、Mosaic + bootstrap風の処理、時点整合を検査するための実装テストである。

## 0. モードと時点整合

分析を3モードに分ける。

- `paper_pre_release`: 文献 Exhibit 7 の pre-release z-score 対応。`release_date - 60 calendar days` から `release_date` の直前まで。構成銘柄は ex post に使うため `is_ex_post=True`。
- `paper_post_release`: 文献 Exhibit 7 の post-release z-score 対応。`release_date` から `release_date + 60 calendar days` まで。60営業日ではなく60カレンダー日で区切り、実計算はその中の取引日のみ。
- `tradable_rolling`: 投資可能なローリング分析。公開済み構成のみを使い、`analysis_date` までの過去営業日で特徴量を作り、翌営業日以降をラベルにする。

文献準拠版は `paper-like` と明記する。完全な Spector et al. (2024) 実装との差分は関数docstringと `warnings` に残す。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"

@dataclass(frozen=True)
class Config:
    seed: int = 42
    window: int = 60
    future_window: int = 60
    release_calendar_days: int = 60
    block_size: int = 10
    bootstrap_reps_synthetic: int = 8
    bootstrap_reps_real: int = 200
    random_bootstrap_reps_synthetic: int = 5
    random_bootstrap_reps_real: int = 80
    random_baskets_synthetic: int = 5
    random_baskets_real: int = 50
    min_obs_ratio: float = 0.80
    z_threshold: float = 2.0
    trading_days_per_year: int = 252
    max_analysis_dates_per_theme: int = 2
    min_stocks_per_factor_multiplier: int = 5
    synthetic_dgp: Literal["apwc_only", "apwc_with_momentum"] = "apwc_only"

cfg = Config()
rng = np.random.default_rng(cfg.seed)
print(cfg)

## 1. 入力スキーマとリターン構築

文献は daily excess returns を使う。そのため `stock_excess_returns(date, ticker, excess_return)` を優先入力とする。後方互換として `stock_total_returns(total_return)` も読むが、cash return が無ければ文献再現readyとは扱わない。

In [ ]:
CORE_SCHEMAS: dict[str, set[str]] = {
    "theme_constituents": {"as_of_date", "theme_id", "theme_name", "release_date", "ticker", "weight"},
    "barra_gemltl_exposures": {"date", "ticker", "factor", "exposure"},
    "universe": {"date", "ticker", "in_universe"},
}
RETURN_SCHEMAS: dict[str, set[str]] = {
    "stock_excess_returns": {"date", "ticker", "excess_return"},
    "stock_total_returns": {"date", "ticker", "total_return"},
}

DATE_COLUMNS = {
    "theme_constituents": ["as_of_date", "release_date"],
    "stock_excess_returns": ["date"],
    "stock_total_returns": ["date"],
    "barra_gemltl_exposures": ["date"],
    "universe": ["date"],
}
NUMERIC_COLUMNS = {
    "theme_constituents": ["weight"],
    "stock_excess_returns": ["excess_return", "cash_return", "benchmark_return"],
    "stock_total_returns": ["total_return", "cash_return", "benchmark_return"],
    "barra_gemltl_exposures": ["exposure"],
    "universe": [],
}


def find_table(stem: str) -> Path | None:
    for suffix in (".parquet", ".csv"):
        candidate = DATA_RAW / f"{stem}{suffix}"
        if candidate.exists():
            return candidate
    return None


def read_table(path: Path) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported table format: {path}")


def validate_schema(name: str, df: pd.DataFrame, required: set[str]) -> pd.DataFrame:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {sorted(missing)}")
    out = df.copy()
    for col in DATE_COLUMNS[name]:
        out[col] = pd.to_datetime(out[col])
    for col in NUMERIC_COLUMNS[name]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    if name == "universe":
        out["in_universe"] = out["in_universe"].astype(bool)
    return out


def prepare_return_table(return_name: str, df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, Any]]:
    out = df.copy()
    meta: dict[str, Any]
    if return_name == "stock_excess_returns":
        out["model_return"] = out["excess_return"]
        meta = {
            "return_type": "excess_return",
            "return_construction": "input_excess_return",
            "paper_replication_ready": True,
            "paper_replication_reason": "Using excess_return",
        }
    else:
        if "cash_return" in out.columns:
            out["excess_return"] = out["total_return"] - out["cash_return"]
            out["model_return"] = out["excess_return"]
            meta = {
                "return_type": "excess_return",
                "return_construction": "total_return_minus_cash_return",
                "paper_replication_ready": True,
                "paper_replication_reason": "Constructed excess_return from total_return - cash_return",
            }
        elif "benchmark_return" in out.columns:
            out["active_return"] = out["total_return"] - out["benchmark_return"]
            out["model_return"] = out["active_return"]
            meta = {
                "return_type": "active_return",
                "return_construction": "total_return_minus_benchmark_return",
                "paper_replication_ready": False,
                "paper_replication_reason": "Using active_return, not excess_return",
            }
        else:
            out["model_return"] = out["total_return"]
            meta = {
                "return_type": "total_return",
                "return_construction": "input_total_return_no_cash_or_benchmark",
                "paper_replication_ready": False,
                "paper_replication_reason": "Using total_return, not excess_return",
            }
    return out, meta

schema_table = pd.DataFrame(
    [(name, sorted(cols)) for name, cols in {**CORE_SCHEMAS, **RETURN_SCHEMAS}.items()],
    columns=["table", "required_columns"],
)
display(schema_table)

## 2. データロードまたは合成データ生成

合成データは2種類を用意する。

- `apwc_only`: APWC検出の smoke test。文献 Exhibit 9/10 のようなリターン持続性を期待しない。
- `apwc_with_momentum`: 共通テーマショックにAR成分を入れ、持続性検査の挙動も確認できる拡張DGP。

デフォルトは `apwc_only`。

In [ ]:
def generate_synthetic_tables(cfg: Config, dgp: str = "apwc_only") -> dict[str, pd.DataFrame]:
    local_rng = np.random.default_rng(cfg.seed)
    dates = pd.bdate_range("2022-01-04", periods=360)
    tickers = [f"JP{i:04d}" for i in range(1, 121)]
    factors = ["market", "value", "size", "momentum", "quality"]

    static_exposure = pd.DataFrame(
        local_rng.normal(0.0, 1.0, size=(len(tickers), len(factors))),
        index=tickers,
        columns=factors,
    )
    static_exposure["market"] = 1.0  # interceptとの完全共線性をrank checkで検出するため、意図的に定数にする。

    exposure_rows = []
    for date in dates:
        drift = local_rng.normal(0.0, 0.015, size=static_exposure.shape)
        exp = static_exposure + drift
        exp["market"] = 1.0
        for ticker in tickers:
            for factor in factors:
                exposure_rows.append((date, ticker, factor, exp.loc[ticker, factor]))
    barra_gemltl_exposures = pd.DataFrame(exposure_rows, columns=["date", "ticker", "factor", "exposure"])

    factor_returns = pd.DataFrame(
        local_rng.normal(0.0, 0.003, size=(len(dates), len(factors))),
        index=dates,
        columns=factors,
    )
    factor_returns["market"] = local_rng.normal(0.0002, 0.006, size=len(dates))

    theme_specs = {
        "T001": {
            "theme_name": "Synthetic Coherent AI Infrastructure",
            "members": tickers[:12],
            "release_idx": 80,
            "common_scale": 0.018,
        },
        "T002": {
            "theme_name": "Synthetic Moderate Green Transition",
            "members": tickers[12:26],
            "release_idx": 95,
            "common_scale": 0.006,
        },
        "T003": {
            "theme_name": "Synthetic Non Coherent Domestic Demand",
            "members": tickers[80:92],
            "release_idx": 190,
            "common_scale": 0.000,
        },
    }

    event_start, event_end = 70, 185
    phi = 0.70 if dgp == "apwc_with_momentum" else 0.00
    drift = 0.0008 if dgp == "apwc_with_momentum" else 0.0
    common_shocks: dict[str, np.ndarray] = {}
    for theme_id, spec in theme_specs.items():
        shock = np.zeros(len(dates))
        for t in range(1, len(dates)):
            scale = spec["common_scale"] if event_start <= t <= event_end else spec["common_scale"] * 0.10
            shock[t] = phi * shock[t - 1] + drift * np.sign(shock[t - 1]) + local_rng.normal(0.0, scale)
        common_shocks[theme_id] = shock

    return_rows = []
    for t, date in enumerate(dates):
        exp_today = barra_gemltl_exposures[barra_gemltl_exposures["date"].eq(date)].pivot(
            index="ticker", columns="factor", values="exposure"
        )[factors]
        systematic = exp_today.to_numpy() @ factor_returns.loc[date, factors].to_numpy()
        residual = local_rng.normal(0.0, 0.008, size=len(tickers))
        for theme_id, spec in theme_specs.items():
            member_idx = [tickers.index(x) for x in spec["members"]]
            residual[member_idx] += common_shocks[theme_id][t]
        excess_return = systematic + residual
        return_rows.extend((date, ticker, ret) for ticker, ret in zip(tickers, excess_return))
    stock_excess_returns = pd.DataFrame(return_rows, columns=["date", "ticker", "excess_return"])

    constituent_rows = []
    for theme_id, spec in theme_specs.items():
        release_date = dates[spec["release_idx"]]
        weight = 1.0 / len(spec["members"])
        for ticker in spec["members"]:
            constituent_rows.append((release_date, theme_id, spec["theme_name"], release_date, ticker, weight))
    theme_constituents = pd.DataFrame(
        constituent_rows,
        columns=["as_of_date", "theme_id", "theme_name", "release_date", "ticker", "weight"],
    )

    universe = pd.MultiIndex.from_product([dates, tickers], names=["date", "ticker"]).to_frame(index=False)
    universe["in_universe"] = True
    return {
        "theme_constituents": theme_constituents,
        "stock_excess_returns": stock_excess_returns,
        "barra_gemltl_exposures": barra_gemltl_exposures,
        "universe": universe,
    }


def load_or_generate_tables(cfg: Config) -> tuple[dict[str, pd.DataFrame], bool, dict[str, Path | None], dict[str, Any]]:
    paths = {name: find_table(name) for name in {**CORE_SCHEMAS, **RETURN_SCHEMAS}}
    present = {name: path for name, path in paths.items() if path is not None}
    if len(present) == 0:
        tables = generate_synthetic_tables(cfg, dgp=cfg.synthetic_dgp)
        validated = {
            "theme_constituents": validate_schema("theme_constituents", tables["theme_constituents"], CORE_SCHEMAS["theme_constituents"]),
            "stock_excess_returns": validate_schema("stock_excess_returns", tables["stock_excess_returns"], RETURN_SCHEMAS["stock_excess_returns"]),
            "barra_gemltl_exposures": validate_schema("barra_gemltl_exposures", tables["barra_gemltl_exposures"], CORE_SCHEMAS["barra_gemltl_exposures"]),
            "universe": validate_schema("universe", tables["universe"], CORE_SCHEMAS["universe"]),
        }
        stock_returns, return_meta = prepare_return_table("stock_excess_returns", validated.pop("stock_excess_returns"))
        validated["stock_returns"] = stock_returns
        return validated, True, paths, return_meta

    missing_core = [name for name in CORE_SCHEMAS if name not in present]
    return_name = "stock_excess_returns" if "stock_excess_returns" in present else "stock_total_returns" if "stock_total_returns" in present else None
    if missing_core or return_name is None:
        raise FileNotFoundError(
            "実データを使う場合は theme_constituents, barra_gemltl_exposures, universe と "
            "stock_excess_returns または stock_total_returns を配置してください。"
            f" present={sorted(present)}, missing_core={missing_core}, data_dir={DATA_RAW}"
        )

    validated = {
        name: validate_schema(name, read_table(present[name]), CORE_SCHEMAS[name])
        for name in CORE_SCHEMAS
    }
    return_required = RETURN_SCHEMAS[return_name]
    return_df = validate_schema(return_name, read_table(present[return_name]), return_required)
    stock_returns, return_meta = prepare_return_table(return_name, return_df)
    validated["stock_returns"] = stock_returns
    return validated, False, paths, return_meta


tables, USING_SYNTHETIC_DATA, input_paths, return_meta = load_or_generate_tables(cfg)
RETURN_TYPE = return_meta["return_type"]
RETURN_CONSTRUCTION = return_meta["return_construction"]
PAPER_REPLICATION_READY = bool(return_meta["paper_replication_ready"])
PAPER_REPLICATION_REASON = return_meta["paper_replication_reason"]

print(f"USING_SYNTHETIC_DATA = {USING_SYNTHETIC_DATA}")
print(f"RETURN_TYPE = {RETURN_TYPE}")
print(f"RETURN_CONSTRUCTION = {RETURN_CONSTRUCTION}")
print(f"PAPER_REPLICATION_READY = {PAPER_REPLICATION_READY}: {PAPER_REPLICATION_REASON}")
display(pd.DataFrame({"table": list(input_paths), "path": [str(v) if v else None for v in input_paths.values()]}))
display(pd.DataFrame({name: [df.shape[0], df.shape[1]] for name, df in tables.items()}, index=["rows", "cols"]).T)

In [ ]:
theme_constituents = tables["theme_constituents"]
stock_returns = tables["stock_returns"]
barra_gemltl_exposures = tables["barra_gemltl_exposures"]
universe = tables["universe"]

schema_checks = []
for name, df in tables.items():
    date_col = "date" if "date" in df.columns else "as_of_date"
    schema_checks.append(
        {
            "table": name,
            "rows": len(df),
            "unique_dates": df[date_col].nunique(),
            "unique_tickers": df["ticker"].nunique() if "ticker" in df.columns else None,
            "missing_cells": int(df.isna().sum().sum()),
        }
    )
display(pd.DataFrame(schema_checks))
assert set(theme_constituents["ticker"]).issubset(set(stock_returns["ticker"])), "テーマ構成銘柄がリターンに存在しません。"
assert set(stock_returns["ticker"]).intersection(set(barra_gemltl_exposures["ticker"])), "リターンとエクスポージャーに共通銘柄がありません。"

## 3. Rank check付きファクターモデル残差

日次クロスセクションOLSで `r = Xb + u` を推定する。設計行列はrank checkを行い、interceptと定数market exposureのような完全共線列を落とす。落とした列はdiagnosticsに残す。

In [ ]:
def build_exposure_lookup(exposures: pd.DataFrame, universe: pd.DataFrame) -> tuple[dict[pd.Timestamp, pd.DataFrame], dict[pd.Timestamp, set[str]], list[str]]:
    factor_names = sorted(exposures["factor"].unique())
    exposure_by_date = {
        date: g.pivot(index="ticker", columns="factor", values="exposure").reindex(columns=factor_names)
        for date, g in exposures.groupby("date", sort=True)
    }
    universe_by_date = {date: set(g.loc[g["in_universe"], "ticker"]) for date, g in universe.groupby("date", sort=True)}
    return exposure_by_date, universe_by_date, factor_names


def select_full_rank_design(exposure_df: pd.DataFrame, factor_names: list[str], tol: float = 1e-10) -> tuple[pd.DataFrame, list[str], list[str], bool]:
    design = pd.DataFrame(index=exposure_df.index)
    design["intercept"] = 1.0
    for factor in factor_names:
        design[factor] = exposure_df[factor].astype(float)

    dropped: list[str] = []
    for col in list(design.columns):
        if col == "intercept":
            continue
        values = design[col].dropna().to_numpy(dtype=float)
        if len(values) == 0 or np.nanstd(values) <= tol:
            dropped.append(col)
            design = design.drop(columns=[col])

    rank_deficient = False
    while design.shape[1] > 0:
        rank = np.linalg.matrix_rank(design.to_numpy(dtype=float), tol=tol)
        if rank == design.shape[1]:
            break
        rank_deficient = True
        candidates = [c for c in design.columns if c != "intercept"]
        if not candidates:
            break
        best_col = candidates[-1]
        best_rank = -1
        for col in candidates:
            test = design.drop(columns=[col])
            test_rank = np.linalg.matrix_rank(test.to_numpy(dtype=float), tol=tol)
            if test_rank > best_rank:
                best_rank = test_rank
                best_col = col
        dropped.append(best_col)
        design = design.drop(columns=[best_col])
    used_factors = [c for c in design.columns if c != "intercept"]
    return design, used_factors, dropped, rank_deficient or bool(dropped)


def fit_residual_wide_from_returns_wide(
    returns_wide: pd.DataFrame,
    exposure_by_date: dict[pd.Timestamp, pd.DataFrame],
    universe_by_date: dict[pd.Timestamp, set[str]],
    factor_names: list[str],
    min_stocks_per_factor_multiplier: int = 1,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    residual_wide = pd.DataFrame(index=returns_wide.index, columns=returns_wide.columns, dtype=float)
    beta_rows: list[dict[str, Any]] = []
    diag_rows: list[dict[str, Any]] = []

    for date in returns_wide.index:
        y_all = returns_wide.loc[date].dropna()
        exposures = exposure_by_date.get(pd.Timestamp(date))
        universe_set = universe_by_date.get(pd.Timestamp(date), set(y_all.index))
        if exposures is None:
            diag_rows.append({"date": date, "status": "missing_exposure"})
            continue
        available = y_all.index.intersection(exposures.dropna().index)
        tickers = [ticker for ticker in available if ticker in universe_set]
        exposure_slice = exposures.loc[tickers, factor_names].dropna()
        tickers = list(y_all.loc[exposure_slice.index].dropna().index)
        exposure_slice = exposure_slice.loc[tickers]
        min_obs = max(2, min_stocks_per_factor_multiplier * (len(factor_names) + 1))
        if len(tickers) < min_obs:
            diag_rows.append({"date": date, "n_obs": len(tickers), "status": "too_few_obs", "dropped_cols": ""})
            continue

        design, used_factors, dropped_cols, rank_issue = select_full_rank_design(exposure_slice, factor_names)
        if design.shape[1] < 1 or len(tickers) <= design.shape[1]:
            diag_rows.append({"date": date, "n_obs": len(tickers), "status": "rank_unusable", "dropped_cols": ";".join(dropped_cols)})
            continue

        y = y_all.loc[tickers].astype(float).to_numpy()
        x = design.loc[tickers].astype(float).to_numpy()
        beta, *_ = np.linalg.lstsq(x, y, rcond=None)
        fitted = x @ beta
        residual = y - fitted
        residual_wide.loc[date, tickers] = residual

        beta_map = dict(zip(design.columns, beta))
        row = {"date": date, "intercept": beta_map.get("intercept", np.nan)}
        row.update({factor: beta_map.get(factor, np.nan) for factor in factor_names})
        beta_rows.append(row)

        factor_inner = {}
        for factor in used_factors:
            factor_inner[factor] = float(abs(np.dot(exposure_slice.loc[tickers, factor].to_numpy(dtype=float), residual)))
        max_inner = max(factor_inner.values()) if factor_inner else 0.0
        diag_rows.append(
            {
                "date": date,
                "n_obs": len(tickers),
                "n_design_cols": design.shape[1],
                "used_factors": ";".join(used_factors),
                "dropped_cols": ";".join(dropped_cols),
                "rank_issue": bool(rank_issue),
                "design_rank": int(np.linalg.matrix_rank(x)),
                "residual_mean": float(np.mean(residual)),
                "residual_std": float(np.std(residual, ddof=1)),
                "max_abs_factor_residual_inner_product": float(max_inner),
                "status": "ok",
            }
        )

    factor_returns = pd.DataFrame(beta_rows)
    if not factor_returns.empty:
        factor_returns = factor_returns.set_index("date").sort_index()
    diagnostics = pd.DataFrame(diag_rows).sort_values("date")
    return residual_wide, factor_returns, diagnostics

returns_wide = stock_returns.pivot(index="date", columns="ticker", values="model_return").sort_index()
exposure_by_date, universe_by_date, factor_names = build_exposure_lookup(barra_gemltl_exposures, universe)
residuals_wide, factor_returns, regression_diagnostics = fit_residual_wide_from_returns_wide(
    returns_wide, exposure_by_date, universe_by_date, factor_names
)
ok_diag = regression_diagnostics[regression_diagnostics["status"].eq("ok")]
max_abs_residual_mean = float(ok_diag["residual_mean"].abs().max())
max_abs_factor_residual_inner_product = float(ok_diag["max_abs_factor_residual_inner_product"].abs().max())
factor_model_rank_issues = int(ok_diag["rank_issue"].sum())
print(f"factors = {factor_names}")
print(f"regression dates ok = {len(ok_diag)} / {len(regression_diagnostics)}")
print(f"rank issue dates = {factor_model_rank_issues}")
print(f"max abs daily residual mean = {max_abs_residual_mean:.3e}")
print(f"max abs factor-residual inner product = {max_abs_factor_residual_inner_product:.3e}")
assert max_abs_residual_mean < 1e-10
assert max_abs_factor_residual_inner_product < 1e-8
display(ok_diag.tail())

## 4. APWCと単体検査

APWCは残差相関行列の上三角平均である。以降のz-score計算に入る前に、基本ケースをNotebook内でassertする。

In [ ]:
def average_pairwise_corr(window: pd.DataFrame, min_obs_ratio: float = cfg.min_obs_ratio) -> float:
    if window.empty:
        return np.nan
    min_obs = int(np.ceil(len(window) * min_obs_ratio))
    clean = window.dropna(axis=1, thresh=min_obs)
    if clean.shape[1] < 2 or clean.shape[0] < 3:
        return np.nan
    corr = clean.corr(min_periods=min_obs)
    values = corr.to_numpy(dtype=float)
    upper = values[np.triu_indices_from(values, k=1)]
    upper = upper[np.isfinite(upper)]
    if upper.size == 0:
        return np.nan
    return float(np.mean(upper))


def run_apwc_unit_tests() -> bool:
    test_rng = np.random.default_rng(123)
    x = pd.Series(np.arange(20, dtype=float))
    y = x * 2.0 + 1.0
    two = pd.DataFrame({"A": x, "B": y})
    assert np.isclose(average_pairwise_corr(two), two.corr().iloc[0, 1])

    identical = pd.DataFrame({"A": x, "B": x, "C": x})
    assert average_pairwise_corr(identical) > 0.999999

    independent = pd.DataFrame(test_rng.normal(size=(400, 8)))
    assert abs(average_pairwise_corr(independent)) < 0.10

    missing = pd.DataFrame({"A": x, "B": y, "C": x})
    missing.loc[:18, "C"] = np.nan
    assert np.isclose(average_pairwise_corr(missing, min_obs_ratio=0.80), two.corr().iloc[0, 1])

    one_valid = pd.DataFrame({"A": x, "B": np.nan})
    assert np.isnan(average_pairwise_corr(one_valid))
    return True

apwc_unit_tests_passed = run_apwc_unit_tests()
print(f"apwc_unit_tests_passed = {apwc_unit_tests_passed}")

## 5. Mosaic + bootstrap: 簡易版とpaper-like版

`block_bootstrap_apwc_simplified` は既存Notebook由来の簡易版であり、文献のMosaic + bootstrapではない。文献対応の主計算では、tile別残差化を行う `mosaic_bootstrap_apwc_paper_like` を使う。

In [ ]:
def latest_constituents(constituents: pd.DataFrame, theme_id: str, analysis_date: pd.Timestamp, public_only: bool = True) -> pd.DataFrame:
    df = constituents[constituents["theme_id"].eq(theme_id)].copy()
    df = df[df["as_of_date"].le(analysis_date)]
    if public_only:
        df = df[df["release_date"].le(analysis_date)]
    if df.empty:
        return df
    latest_as_of = df["as_of_date"].max()
    df = df[df["as_of_date"].eq(latest_as_of)].copy()
    weight_sum = df["weight"].sum()
    df["weight"] = 1.0 / len(df) if not np.isfinite(weight_sum) or abs(weight_sum) < 1e-12 else df["weight"] / weight_sum
    return df


def basket_residual_return(residual_window: pd.DataFrame, constituent_df: pd.DataFrame) -> pd.Series:
    tickers = [ticker for ticker in constituent_df["ticker"] if ticker in residual_window.columns]
    if len(tickers) == 0:
        return pd.Series(index=residual_window.index, dtype=float)
    weights = constituent_df.set_index("ticker").loc[tickers, "weight"].astype(float)
    weights = weights / weights.sum()
    return residual_window[tickers].mul(weights, axis=1).sum(axis=1, min_count=1)


def contiguous_blocks(n: int, block_size: int) -> list[np.ndarray]:
    return [np.arange(start, min(start + block_size, n)) for start in range(0, n, block_size)]


def block_permute_columns_by_time(residual_window: pd.DataFrame, block_size: int, rng: np.random.Generator) -> pd.DataFrame:
    out = pd.DataFrame(index=residual_window.index, columns=residual_window.columns, dtype=float)
    blocks = contiguous_blocks(len(residual_window), block_size)
    for ticker in residual_window.columns:
        values = residual_window[ticker].to_numpy(dtype=float)
        order = rng.permutation(len(blocks))
        out[ticker] = np.concatenate([values[blocks[i]] for i in order])[: len(values)]
    return out


def block_bootstrap_apwc_simplified(
    residuals_wide: pd.DataFrame,
    dates: pd.Index,
    basket_tickers: list[str],
    n_bootstrap: int,
    seed: int,
    block_size: int = 10,
) -> dict[str, Any]:
    """文献のMosaic + bootstrapの簡易版。

    全ユニバース一括残差化済みのresidualをblock shuffleするだけで、文献のtile別残差化とは異なる。
    回帰残差の人工構造を十分に緩和できず、z-scoreを過大評価する可能性がある。
    本番の文献再現値としては使わない。
    """
    local_rng = np.random.default_rng(seed)
    window = residuals_wide.loc[dates]
    observed = average_pairwise_corr(window[basket_tickers])
    boot_values = []
    for _ in range(n_bootstrap):
        shuffled = block_permute_columns_by_time(window, block_size, local_rng)
        boot_values.append(average_pairwise_corr(shuffled[basket_tickers]))
    boot = np.array([x for x in boot_values if np.isfinite(x)], dtype=float)
    null_mean = float(np.mean(boot)) if len(boot) else np.nan
    null_std = float(np.std(boot, ddof=1)) if len(boot) > 1 else np.nan
    z = (observed - null_mean) / null_std if null_std and np.isfinite(null_std) else np.nan
    return {
        "observed_apwc": observed,
        "z_score": z,
        "null_mean": null_mean,
        "null_std": null_std,
        "n_bootstrap": int(len(boot)),
        "method": "block_bootstrap_simplified_not_for_paper_replication",
        "warnings": ["Simplified residual block shuffle; not paper-like Mosaic + bootstrap."],
    }


def trading_dates_between(index: pd.Index, start: pd.Timestamp, end: pd.Timestamp) -> pd.Index:
    idx = pd.Index(pd.to_datetime(index)).sort_values()
    return idx[(idx >= pd.Timestamp(start)) & (idx <= pd.Timestamp(end))]


def make_mosaic_tiles(
    dates: pd.Index,
    returns_wide: pd.DataFrame,
    exposure_by_date: dict[pd.Timestamp, pd.DataFrame],
    universe_by_date: dict[pd.Timestamp, set[str]],
    factor_names: list[str],
    block_size: int,
    seed: int,
    min_stocks_per_factor_multiplier: int,
) -> tuple[list[dict[str, Any]], list[str]]:
    local_rng = np.random.default_rng(seed)
    warnings: list[str] = []
    tiles: list[dict[str, Any]] = []
    p = len(factor_names) + 1
    min_group_size = max(2, min_stocks_per_factor_multiplier * p)
    date_blocks = contiguous_blocks(len(dates), block_size)
    for batch_id, block in enumerate(date_blocks):
        batch_dates = pd.Index(dates[block])
        valid_sets = []
        for date in batch_dates:
            y_valid = set(returns_wide.loc[date].dropna().index)
            exp = exposure_by_date.get(pd.Timestamp(date))
            exp_valid = set(exp.dropna().index) if exp is not None else set()
            uni_valid = universe_by_date.get(pd.Timestamp(date), y_valid)
            valid_sets.append(y_valid & exp_valid & uni_valid)
        if not valid_sets:
            continue
        valid_tickers = sorted(set.intersection(*valid_sets))
        if len(valid_tickers) < min_group_size:
            warnings.append(f"batch {batch_id}: too few stocks for 5*p grouping; using one small group n={len(valid_tickers)}")
            k = 1
        else:
            k = max(1, len(valid_tickers) // min_group_size)
        shuffled = np.array(valid_tickers, dtype=object)
        local_rng.shuffle(shuffled)
        groups = [list(g) for g in np.array_split(shuffled, k) if len(g) > 0]
        for group_id, group in enumerate(groups):
            tiles.append({"batch_id": batch_id, "group_id": group_id, "dates": batch_dates, "tickers": group})
    return tiles, warnings


def residualize_by_tiles(
    returns_subset: pd.DataFrame,
    tiles: list[dict[str, Any]],
    exposure_by_date: dict[pd.Timestamp, pd.DataFrame],
    universe_by_date: dict[pd.Timestamp, set[str]],
    factor_names: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    residual = pd.DataFrame(index=returns_subset.index, columns=returns_subset.columns, dtype=float)
    fitted = pd.DataFrame(index=returns_subset.index, columns=returns_subset.columns, dtype=float)
    diagnostics = []
    for tile in tiles:
        tile_dates = pd.Index([d for d in tile["dates"] if d in returns_subset.index])
        tile_tickers = [t for t in tile["tickers"] if t in returns_subset.columns]
        if len(tile_dates) == 0 or len(tile_tickers) == 0:
            continue
        tile_returns = returns_subset.loc[tile_dates, tile_tickers]
        tile_resid, _, tile_diag = fit_residual_wide_from_returns_wide(
            tile_returns,
            exposure_by_date,
            universe_by_date,
            factor_names,
            min_stocks_per_factor_multiplier=1,
        )
        residual.loc[tile_dates, tile_tickers] = tile_resid.loc[tile_dates, tile_tickers]
        fitted.loc[tile_dates, tile_tickers] = tile_returns - tile_resid
        if not tile_diag.empty:
            tile_diag = tile_diag.copy()
            tile_diag["batch_id"] = tile["batch_id"]
            tile_diag["group_id"] = tile["group_id"]
            tile_diag["tile_n_tickers"] = len(tile_tickers)
            diagnostics.append(tile_diag)
    diag = pd.concat(diagnostics, ignore_index=True) if diagnostics else pd.DataFrame()
    return residual, fitted, diag


def mosaic_bootstrap_apwc_paper_like(
    returns_wide: pd.DataFrame,
    exposure_by_date: dict[pd.Timestamp, pd.DataFrame],
    universe_by_date: dict[pd.Timestamp, set[str]],
    factor_names: list[str],
    basket_tickers: list[str],
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
    block_size: int,
    n_bootstrap: int,
    seed: int,
    min_obs_ratio: float,
    min_stocks_per_factor_multiplier: int = 5,
) -> dict[str, Any]:
    """文献のMosaic + bootstrapに近づけたpaper-like実装。

    10連続観測日のtime batchとstock groupを作り、各time x stock tile内で日次クロスセクション回帰を別々に実行する。
    bootstrapではtile残差を10日block単位でscrambleし、synthetic returnを同じtile構造で再残差化する。

    完全なSpector et al. (2024)実装ではない。特にlocal exchangeabilityの厳密な検定手順、商用Barra推奨ウェイト、実際のMosaic permutationの全詳細は未実装である。
    そのため出力methodは `mosaic_bootstrap_paper_like` とし、warningsに差分を残す。
    """
    dates = trading_dates_between(returns_wide.index, window_start, window_end)
    warnings: list[str] = []
    if len(dates) < 3:
        return {
            "observed_apwc": np.nan,
            "z_score": np.nan,
            "null_mean": np.nan,
            "null_std": np.nan,
            "n_bootstrap": 0,
            "n_valid_dates": int(len(dates)),
            "n_valid_basket_tickers": 0,
            "method": "mosaic_bootstrap_paper_like",
            "tile_diagnostics": pd.DataFrame(),
            "warnings": ["Too few valid trading dates in window."],
        }

    candidate_basket = [t for t in basket_tickers if t in returns_wide.columns]
    tiles, tile_warnings = make_mosaic_tiles(
        dates, returns_wide, exposure_by_date, universe_by_date, factor_names, block_size, seed, min_stocks_per_factor_multiplier
    )
    warnings.extend(tile_warnings)
    returns_subset = returns_wide.loc[dates]
    tile_resid, tile_fitted, tile_diag = residualize_by_tiles(returns_subset, tiles, exposure_by_date, universe_by_date, factor_names)
    valid_basket = [t for t in candidate_basket if t in tile_resid.columns and tile_resid[t].notna().sum() >= int(np.ceil(len(dates) * min_obs_ratio))]
    observed_apwc = average_pairwise_corr(tile_resid[valid_basket], min_obs_ratio=min_obs_ratio) if len(valid_basket) >= 2 else np.nan

    local_rng = np.random.default_rng(seed + 991)
    boot_values = []
    for _ in range(n_bootstrap):
        shuffled_resid = block_permute_columns_by_time(tile_resid, block_size, local_rng)
        synthetic_returns = tile_fitted + shuffled_resid
        boot_resid, _, _ = residualize_by_tiles(synthetic_returns, tiles, exposure_by_date, universe_by_date, factor_names)
        boot_apwc = average_pairwise_corr(boot_resid[valid_basket], min_obs_ratio=min_obs_ratio) if len(valid_basket) >= 2 else np.nan
        if np.isfinite(boot_apwc):
            boot_values.append(float(boot_apwc))

    boot = np.array(boot_values, dtype=float)
    null_mean = float(np.mean(boot)) if len(boot) else np.nan
    null_std = float(np.std(boot, ddof=1)) if len(boot) > 1 else np.nan
    z_score = (observed_apwc - null_mean) / null_std if null_std and np.isfinite(null_std) else np.nan
    warnings.append("Paper-like implementation: tile residualization is implemented, but exact Spector et al. local exchangeability procedure is not fully reproduced.")
    return {
        "observed_apwc": float(observed_apwc) if np.isfinite(observed_apwc) else np.nan,
        "z_score": float(z_score) if np.isfinite(z_score) else np.nan,
        "null_mean": null_mean,
        "null_std": null_std,
        "n_bootstrap": int(len(boot)),
        "n_valid_dates": int(len(dates)),
        "n_valid_basket_tickers": int(len(valid_basket)),
        "method": "mosaic_bootstrap_paper_like",
        "tile_diagnostics": tile_diag,
        "warnings": warnings,
    }

## 6. Release前後60カレンダー日のpaperモード

文献 Exhibit 7 に対応する。pre-releaseは公開前に構成銘柄が利用可能だったとは仮定しないため、`is_ex_post=True` を明記する。

In [ ]:
def compute_release_window_zscores(
    constituents: pd.DataFrame,
    returns_wide: pd.DataFrame,
    exposure_by_date: dict[pd.Timestamp, pd.DataFrame],
    universe_by_date: dict[pd.Timestamp, set[str]],
    factor_names: list[str],
    cfg: Config,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    n_bootstrap = cfg.bootstrap_reps_synthetic if USING_SYNTHETIC_DATA else cfg.bootstrap_reps_real
    for theme_id, g in constituents.groupby("theme_id", sort=True):
        release_date = pd.Timestamp(g["release_date"].min())
        basket = latest_constituents(constituents, theme_id, release_date, public_only=False)
        basket_tickers = list(basket["ticker"])
        pre_start = release_date - pd.Timedelta(days=cfg.release_calendar_days)
        pre_end = release_date - pd.Timedelta(days=1)
        post_start = release_date
        post_end = release_date + pd.Timedelta(days=cfg.release_calendar_days)
        seed_base = cfg.seed + sum(ord(ch) for ch in str(theme_id))
        pre = mosaic_bootstrap_apwc_paper_like(
            returns_wide, exposure_by_date, universe_by_date, factor_names, basket_tickers,
            pre_start, pre_end, cfg.block_size, n_bootstrap, seed_base + 11, cfg.min_obs_ratio, cfg.min_stocks_per_factor_multiplier,
        )
        post = mosaic_bootstrap_apwc_paper_like(
            returns_wide, exposure_by_date, universe_by_date, factor_names, basket_tickers,
            post_start, post_end, cfg.block_size, n_bootstrap, seed_base + 29, cfg.min_obs_ratio, cfg.min_stocks_per_factor_multiplier,
        )
        rows.append(
            {
                "theme_id": theme_id,
                "theme_name": g["theme_name"].iloc[0],
                "release_date": release_date,
                "n_constituents": len(basket_tickers),
                "pre_start": pre_start,
                "pre_end": pre_end,
                "post_start": post_start,
                "post_end": post_end,
                "pre_apwc": pre["observed_apwc"],
                "pre_z_score": pre["z_score"],
                "post_apwc": post["observed_apwc"],
                "post_z_score": post["z_score"],
                "pre_is_coherent": bool(pre["z_score"] >= cfg.z_threshold) if np.isfinite(pre["z_score"]) else False,
                "post_is_coherent": bool(post["z_score"] >= cfg.z_threshold) if np.isfinite(post["z_score"]) else False,
                "method": "mosaic_bootstrap_paper_like",
                "is_ex_post": True,
                "mode_pre": "paper_pre_release",
                "mode_post": "paper_post_release",
                "warnings": " | ".join(sorted(set(pre["warnings"] + post["warnings"]))),
            }
        )
    result = pd.DataFrame(rows)
    summary = pd.DataFrame(
        [
            {
                "n_themes": len(result),
                "pre_coherent_count": int(result["pre_is_coherent"].sum()),
                "pre_coherent_share": float(result["pre_is_coherent"].mean()) if len(result) else np.nan,
                "post_coherent_count": int(result["post_is_coherent"].sum()),
                "post_coherent_share": float(result["post_is_coherent"].mean()) if len(result) else np.nan,
                "pre_post_z_corr": float(result[["pre_z_score", "post_z_score"]].corr().iloc[0, 1]) if len(result) > 1 else np.nan,
            }
        ]
    )
    return result, summary

release_window_results, release_window_summary = compute_release_window_zscores(
    theme_constituents, returns_wide, exposure_by_date, universe_by_date, factor_names, cfg
)
release_window_mode_available = True
display(release_window_results[["theme_id", "release_date", "pre_apwc", "pre_z_score", "post_apwc", "post_z_score", "pre_is_coherent", "post_is_coherent", "is_ex_post"]])
display(release_window_summary)
assert (release_window_results["pre_end"] == release_window_results["release_date"] - pd.Timedelta(days=1)).all()
assert (release_window_results["post_end"] == release_window_results["release_date"] + pd.Timedelta(days=cfg.release_calendar_days)).all()

## 7. Tradable rolling モード

公開済み構成のみを使う投資可能分析。特徴量は `analysis_date` まで、ラベルは翌営業日以降とする。

In [ ]:
def choose_tradable_analysis_dates(constituents: pd.DataFrame, trading_dates: pd.Index, cfg: Config) -> pd.DataFrame:
    rows = []
    trading_dates = pd.Index(pd.to_datetime(trading_dates)).sort_values()
    for theme_id, g in constituents.groupby("theme_id", sort=True):
        release_date = pd.Timestamp(g["release_date"].min())
        release_pos = int(np.searchsorted(trading_dates.values, np.datetime64(release_date), side="left"))
        first_pos = release_pos + cfg.window
        last_pos = len(trading_dates) - cfg.future_window - 1
        if first_pos > last_pos:
            continue
        candidates = trading_dates[first_pos : last_pos + 1]
        if len(candidates) <= cfg.max_analysis_dates_per_theme:
            selected = candidates
        else:
            selected_idx = np.linspace(0, len(candidates) - 1, cfg.max_analysis_dates_per_theme).round().astype(int)
            selected = candidates[selected_idx]
        for analysis_date in selected:
            rows.append(
                {
                    "mode": "tradable_rolling",
                    "theme_id": theme_id,
                    "theme_name": g["theme_name"].iloc[0],
                    "release_date": release_date,
                    "analysis_date": pd.Timestamp(analysis_date),
                }
            )
    return pd.DataFrame(rows)


def analyze_tradable_theme_date(row: pd.Series, cfg: Config) -> dict[str, Any]:
    theme_id = row["theme_id"]
    analysis_date = pd.Timestamp(row["analysis_date"])
    all_dates = residuals_wide.index
    pos = all_dates.get_loc(analysis_date)
    past_dates = all_dates[pos - cfg.window + 1 : pos + 1]
    future_dates = all_dates[pos + 1 : pos + 1 + cfg.future_window]
    assert len(past_dates) == cfg.window
    assert len(future_dates) == cfg.future_window
    assert max(past_dates) <= analysis_date
    assert min(future_dates) > analysis_date

    constituents_now = latest_constituents(theme_constituents, theme_id, analysis_date, public_only=True)
    basket_tickers = [ticker for ticker in constituents_now["ticker"] if ticker in returns_wide.columns]
    n_bootstrap = cfg.bootstrap_reps_synthetic if USING_SYNTHETIC_DATA else cfg.bootstrap_reps_real
    boot = mosaic_bootstrap_apwc_paper_like(
        returns_wide, exposure_by_date, universe_by_date, factor_names, basket_tickers,
        pd.Timestamp(past_dates[0]), pd.Timestamp(past_dates[-1]), cfg.block_size, n_bootstrap,
        cfg.seed + int(pos) + sum(ord(ch) for ch in str(theme_id)), cfg.min_obs_ratio, cfg.min_stocks_per_factor_multiplier,
    )
    past_basket = basket_residual_return(residuals_wide.loc[past_dates], constituents_now)
    future_basket = basket_residual_return(residuals_wide.loc[future_dates], constituents_now)
    return {
        "mode": "tradable_rolling",
        "theme_id": theme_id,
        "theme_name": row["theme_name"],
        "release_date": row["release_date"],
        "analysis_date": analysis_date,
        "past_start": pd.Timestamp(past_dates[0]),
        "past_end": pd.Timestamp(past_dates[-1]),
        "future_start": pd.Timestamp(future_dates[0]),
        "future_end": pd.Timestamp(future_dates[-1]),
        "n_members": len(basket_tickers),
        "apwc": boot["observed_apwc"],
        "boot_mean": boot["null_mean"],
        "boot_std": boot["null_std"],
        "z_score": boot["z_score"],
        "is_coherent": bool(boot["z_score"] >= cfg.z_threshold) if np.isfinite(boot["z_score"]) else False,
        "past_residual_return": float(past_basket.sum()),
        "future_residual_return": float(future_basket.sum()),
        "method": boot["method"],
        "warnings": " | ".join(boot["warnings"]),
    }

tradable_calendar = choose_tradable_analysis_dates(theme_constituents, residuals_wide.index, cfg)
tradable_results = pd.DataFrame([analyze_tradable_theme_date(row, cfg) for _, row in tradable_calendar.iterrows()])
tradable_results["feature_max_date"] = tradable_results["past_end"]
tradable_results["label_min_date"] = tradable_results["future_start"]
tradable_rolling_timing_ok = bool((tradable_results["feature_max_date"] <= tradable_results["analysis_date"]).all() and (tradable_results["label_min_date"] > tradable_results["analysis_date"]).all())
assert tradable_rolling_timing_ok
display(tradable_results[["theme_id", "analysis_date", "apwc", "z_score", "is_coherent", "past_residual_return", "future_residual_return"]])

## 8. 簡易bootstrapのデモ

簡易版は本番の文献再現値には使わない。ここでは関数の区別を明示するため、最初のtradable行だけに低repで実行する。

In [ ]:
if len(tradable_results):
    demo_row = tradable_results.iloc[0]
    demo_dates = trading_dates_between(residuals_wide.index, demo_row["past_start"], demo_row["past_end"])
    demo_constituents = latest_constituents(theme_constituents, demo_row["theme_id"], demo_row["analysis_date"], public_only=True)
    simplified_demo = block_bootstrap_apwc_simplified(
        residuals_wide, demo_dates, list(demo_constituents["ticker"]), n_bootstrap=3, seed=cfg.seed + 777, block_size=cfg.block_size
    )
else:
    simplified_demo = {"method": "not_run", "warnings": ["No tradable rows."]}
simplified_bootstrap_used_only_for_demo = simplified_demo["method"] != "not_run"
display(pd.Series({k: v for k, v in simplified_demo.items() if k != "warnings"}, name="simplified_demo"))
print("warnings:", simplified_demo.get("warnings"))

## 9. リスク過小評価検証

現状は実データのidiosyncratic varianceが無いため、過去窓の残差サンプル分散の対角をリスクモデルの固有分散近似として使う。これは in-sample 比較であり、将来窓のout-of-sample実現リスク比較は拡張課題である。

In [ ]:
def residual_risk_comparison(row: pd.Series, cfg: Config, idio_var_source: str = "sample_diag") -> dict[str, Any]:
    """残差相関を無視する対角モデルと、過去窓のfull sample covarianceによる残差リスクを比較する。

    idio_var_source="sample_diag" は過去窓の残差サンプル分散の対角を使う。実データにrisk model由来のidiosyncratic varianceがある場合は
    idio_var_source="risk_model" を拡張実装する余地がある。ここでの比較はin-sampleであり、out-of-sample実現リスクではない。
    """
    if idio_var_source != "sample_diag":
        raise NotImplementedError("risk_model idiosyncratic variance input is not available in this notebook yet.")
    constituents_now = latest_constituents(theme_constituents, row["theme_id"], row["analysis_date"], public_only=True)
    tickers = [ticker for ticker in constituents_now["ticker"] if ticker in residuals_wide.columns]
    weights = constituents_now.set_index("ticker").loc[tickers, "weight"].astype(float)
    weights = weights / weights.sum()
    window = residuals_wide.loc[(residuals_wide.index >= row["past_start"]) & (residuals_wide.index <= row["past_end"]), tickers]
    window = window.dropna(axis=1, thresh=int(np.ceil(len(window) * cfg.min_obs_ratio)))
    tickers = list(window.columns)
    weights = weights.loc[tickers]
    weights = weights / weights.sum()
    cov = window.cov() * cfg.trading_days_per_year
    diag_cov = pd.DataFrame(np.diag(np.diag(cov.to_numpy())), index=cov.index, columns=cov.columns)
    w = weights.to_numpy()
    model_risk = float(np.sqrt(w @ diag_cov.to_numpy() @ w))
    empirical_full_risk = float(np.sqrt(w @ cov.to_numpy() @ w))
    return {
        "theme_id": row["theme_id"],
        "analysis_date": row["analysis_date"],
        "is_coherent": bool(row["is_coherent"]),
        "apwc": row["apwc"],
        "model_residual_risk": model_risk,
        "empirical_full_residual_risk": empirical_full_risk,
        "risk_understatement_ratio": empirical_full_risk / model_risk if model_risk > 0 else np.nan,
        "diag_source": idio_var_source,
        "annualized": True,
    }

risk_table = pd.DataFrame([residual_risk_comparison(row, cfg, idio_var_source="sample_diag") for _, row in tradable_results.iterrows()])
risk_group_summary = risk_table.groupby("is_coherent").agg(
    rows=("theme_id", "size"),
    mean_risk_understatement_ratio=("risk_understatement_ratio", "mean"),
    median_risk_understatement_ratio=("risk_understatement_ratio", "median"),
).reset_index()
display(risk_table)
display(risk_group_summary)

## 10. 残差リターン持続性

合成データのデフォルト `apwc_only` では文献 Exhibit 9/10 と同じ結論を期待しない。文献型の持続性を合成データで検査したい場合は `cfg.synthetic_dgp="apwc_with_momentum"` に切り替える。

In [ ]:
def persistence_regression(results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for coherent_flag, g in results.groupby("is_coherent", sort=True):
        g = g.dropna(subset=["past_residual_return", "future_residual_return"])
        if len(g) >= 3 and g["past_residual_return"].std(ddof=1) > 0:
            x = sm.add_constant(g["past_residual_return"], has_constant="add")
            y = g["future_residual_return"]
            model = sm.OLS(y, x).fit()
            rows.append(
                {
                    "group": "z >= 2" if coherent_flag else "z < 2",
                    "estimated_coefficient": model.params["past_residual_return"],
                    "t_statistic": model.tvalues["past_residual_return"],
                    "observations": int(model.nobs),
                    "r_squared": model.rsquared,
                    "interpretation_ready": not USING_SYNTHETIC_DATA,
                }
            )
        else:
            rows.append(
                {
                    "group": "z >= 2" if coherent_flag else "z < 2",
                    "estimated_coefficient": np.nan,
                    "t_statistic": np.nan,
                    "observations": len(g),
                    "r_squared": np.nan,
                    "interpretation_ready": not USING_SYNTHETIC_DATA,
                }
            )
    return pd.DataFrame(rows)

persistence_table = persistence_regression(tradable_results)
persistence_interpretation_ready = not USING_SYNTHETIC_DATA
display(persistence_table)

## 11. ランダムバスケット calibration

文献では、ランダムバスケットのz-scoreがおおむね0中心、概ね -2 から 2 に収まることをキャリブレーションとして確認している。ここでも同じ方法で負例を作る。

In [ ]:
def random_basket_calibration(
    returns_wide: pd.DataFrame,
    exposure_by_date: dict[pd.Timestamp, pd.DataFrame],
    universe_by_date: dict[pd.Timestamp, set[str]],
    factor_names: list[str],
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
    basket_size: int,
    n_baskets: int,
    cfg: Config,
    seed: int,
    exclude_tickers: set[str] | None = None,
) -> tuple[pd.DataFrame, dict[str, float]]:
    local_rng = np.random.default_rng(seed)
    dates = trading_dates_between(returns_wide.index, window_start, window_end)
    if len(dates) == 0:
        raise ValueError("No trading dates for random basket calibration.")
    eligible = set(returns_wide.columns)
    if exclude_tickers:
        eligible = eligible - set(exclude_tickers)
    eligible = np.array(sorted(eligible), dtype=object)
    n_bootstrap = cfg.random_bootstrap_reps_synthetic if USING_SYNTHETIC_DATA else cfg.random_bootstrap_reps_real
    rows = []
    for i in range(n_baskets):
        sample = list(local_rng.choice(eligible, size=min(basket_size, len(eligible)), replace=False))
        result = mosaic_bootstrap_apwc_paper_like(
            returns_wide, exposure_by_date, universe_by_date, factor_names, sample,
            window_start, window_end, cfg.block_size, n_bootstrap, seed + 1000 + i, cfg.min_obs_ratio, cfg.min_stocks_per_factor_multiplier,
        )
        rows.append({"basket_id": f"random_{i:03d}", "n_tickers": len(sample), "apwc": result["observed_apwc"], "z_score": result["z_score"], "method": result["method"]})
    detail = pd.DataFrame(rows)
    summary = {
        "mean_z": float(detail["z_score"].mean()),
        "std_z": float(detail["z_score"].std(ddof=1)),
        "min_z": float(detail["z_score"].min()),
        "max_z": float(detail["z_score"].max()),
        "share_abs_z_gt_2": float((detail["z_score"].abs() > cfg.z_threshold).mean()),
    }
    return detail, summary

reference_release = release_window_results.iloc[-1]
all_theme_tickers = set(theme_constituents["ticker"])
random_n = cfg.random_baskets_synthetic if USING_SYNTHETIC_DATA else cfg.random_baskets_real
random_detail, random_summary = random_basket_calibration(
    returns_wide, exposure_by_date, universe_by_date, factor_names,
    reference_release["post_start"], reference_release["post_end"], int(reference_release["n_constituents"]),
    random_n, cfg, cfg.seed + 2024, exclude_tickers=all_theme_tickers if USING_SYNTHETIC_DATA else set(),
)
random_basket_calibration_mean_z = random_summary["mean_z"]
random_basket_calibration_share_abs_z_gt_2 = random_summary["share_abs_z_gt_2"]
display(random_detail)
display(pd.Series(random_summary, name="random_basket_calibration"))

## 12. Coherence条件付きモメンタム診断

APWC上位50%に絞ったモメンタム改善が、既存ファクターではなく残差共通ショックに由来するかを確認する追加診断。既存の `tradable_results`、`residuals_wide`、`returns_wide`、`factor_returns` を再利用し、既存のMosaic/release window実装は変更しない。

合成データ時の数値は研究結果ではなく smoke test として扱う。

In [ ]:
def basket_weighted_return(return_window: pd.DataFrame, constituent_df: pd.DataFrame) -> pd.Series:
    tickers = [ticker for ticker in constituent_df["ticker"] if ticker in return_window.columns]
    if len(tickers) == 0:
        return pd.Series(index=return_window.index, dtype=float)
    weights = constituent_df.set_index("ticker").loc[tickers, "weight"].astype(float)
    weights = weights / weights.sum()
    return return_window[tickers].mul(weights, axis=1).sum(axis=1, min_count=1)


def _standardize_within_date(values: pd.Series) -> pd.Series:
    std = values.std(ddof=0)
    if not np.isfinite(std) or std == 0:
        return pd.Series(0.0, index=values.index)
    return (values - values.mean()) / std


def assign_top50_by_date(df: pd.DataFrame, score_col: str) -> pd.Series:
    flags = pd.Series(False, index=df.index)
    for _, g in df.groupby("analysis_date", sort=True):
        n_top = max(1, int(np.ceil(len(g) * 0.50)))
        top_idx = g[score_col].sort_values(ascending=False, na_position="last").head(n_top).index
        flags.loc[top_idx] = True
    return flags


def build_coherence_momentum_panel(tradable_results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in tradable_results.iterrows():
        analysis_date = pd.Timestamp(row["analysis_date"])
        constituents_now = latest_constituents(theme_constituents, row["theme_id"], analysis_date, public_only=True)
        past_dates = trading_dates_between(returns_wide.index, row["past_start"], row["past_end"])
        future_dates = trading_dates_between(returns_wide.index, row["future_start"], row["future_end"])
        assert len(past_dates) > 0
        assert len(future_dates) > 0
        assert pd.Timestamp(past_dates.max()) <= analysis_date
        assert pd.Timestamp(future_dates.min()) > analysis_date
        past_total = basket_weighted_return(returns_wide.loc[past_dates], constituents_now)
        future_total = basket_weighted_return(returns_wide.loc[future_dates], constituents_now)
        past_resid = basket_residual_return(residuals_wide.loc[past_dates], constituents_now)
        future_resid = basket_residual_return(residuals_wide.loc[future_dates], constituents_now)
        rows.append(
            {
                "mode": "coherence_momentum_diagnostic",
                "theme_id": row["theme_id"],
                "theme_name": row["theme_name"],
                "analysis_date": analysis_date,
                "past_start": pd.Timestamp(past_dates.min()),
                "past_end": pd.Timestamp(past_dates.max()),
                "future_start": pd.Timestamp(future_dates.min()),
                "future_end": pd.Timestamp(future_dates.max()),
                "apwc": row["apwc"],
                "z_score": row["z_score"],
                "past_total_momentum": float(past_total.sum()),
                "past_residual_momentum": float(past_resid.sum()),
                "future_total_return": float(future_total.sum()),
                "future_residual_return": float(future_resid.sum()),
                "n_members": row["n_members"],
            }
        )
    panel = pd.DataFrame(rows)
    fallback = panel.groupby("analysis_date")["apwc"].transform(_standardize_within_date)
    panel["coherence_z"] = panel["z_score"].where(panel["z_score"].notna(), fallback)
    panel["coherence_top50"] = assign_top50_by_date(panel, "coherence_z")
    assert panel.groupby("analysis_date")["coherence_top50"].sum().min() >= 1
    assert (panel["past_end"] <= panel["analysis_date"]).all()
    assert (panel["future_start"] > panel["analysis_date"]).all()
    return panel


def simple_t_stat(values: pd.Series) -> float:
    values = pd.Series(values).dropna()
    if len(values) < 2:
        return np.nan
    sd = values.std(ddof=1)
    if not np.isfinite(sd) or sd == 0:
        return np.nan
    return float(values.mean() / (sd / np.sqrt(len(values))))


def select_top_momentum_by_date(df: pd.DataFrame, signal_col: str) -> pd.DataFrame:
    selected = []
    for _, g in df.groupby("analysis_date", sort=True):
        if g.empty:
            continue
        n_pick = max(1, int(np.ceil(len(g) * 0.50)))
        selected.append(g.sort_values(signal_col, ascending=False).head(n_pick))
    if not selected:
        return pd.DataFrame(columns=df.columns)
    return pd.concat(selected, ignore_index=True)


def summarize_selected_momentum(selected: pd.DataFrame, label: str) -> dict[str, float | str | int]:
    return {
        "strategy": label,
        "n_obs": int(len(selected)),
        "future_total_return_mean": float(selected["future_total_return"].mean()) if len(selected) else np.nan,
        "future_total_return_t": simple_t_stat(selected["future_total_return"]),
        "future_total_win_rate": float((selected["future_total_return"] > 0).mean()) if len(selected) else np.nan,
        "future_residual_return_mean": float(selected["future_residual_return"].mean()) if len(selected) else np.nan,
        "future_residual_return_t": simple_t_stat(selected["future_residual_return"]),
    }


def build_strategy_return_panel(panel: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for label, base in [("all_momentum", panel), ("coherence_top50_momentum", panel[panel["coherence_top50"]])]:
        for analysis_date, g in base.groupby("analysis_date", sort=True):
            selected = select_top_momentum_by_date(g, "past_total_momentum")
            if selected.empty:
                continue
            factor_window = factor_returns.loc[(factor_returns.index >= selected["future_start"].min()) & (factor_returns.index <= selected["future_end"].max())]
            row = {
                "strategy": label,
                "analysis_date": analysis_date,
                "future_start": selected["future_start"].min(),
                "future_end": selected["future_end"].max(),
                "strategy_return": float(selected["future_total_return"].mean()),
                "strategy_residual_return": float(selected["future_residual_return"].mean()),
                "n_selected": int(len(selected)),
            }
            for col in factor_returns.columns:
                row[f"factor_{col}"] = float(factor_window[col].sum()) if col in factor_window else np.nan
            rows.append(row)
    return pd.DataFrame(rows)


def compare_momentum_by_coherence(panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    all_selected = select_top_momentum_by_date(panel, "past_total_momentum")
    top50_selected = select_top_momentum_by_date(panel[panel["coherence_top50"]], "past_total_momentum")
    summary = pd.DataFrame(
        [
            summarize_selected_momentum(all_selected, "all_momentum"),
            summarize_selected_momentum(top50_selected, "coherence_top50_momentum"),
        ]
    )
    strategy_panel = build_strategy_return_panel(panel)
    return summary, strategy_panel


def run_factor_decomposition(strategy_panel: pd.DataFrame) -> pd.DataFrame:
    factor_cols = [c for c in strategy_panel.columns if c.startswith("factor_")]
    rows = []
    for strategy, g in strategy_panel.groupby("strategy", sort=True):
        g = g.dropna(subset=["strategy_return"] + factor_cols)
        if len(g) <= 2 or len(factor_cols) == 0:
            rows.append({"strategy": strategy, "alpha": np.nan, "alpha_t": np.nan, "factor_beta_norm": np.nan, "r_squared": np.nan, "n_obs": int(len(g)), "status": "too_few_obs"})
            continue
        x_raw = g[factor_cols].astype(float)
        keep_cols = [c for c in factor_cols if x_raw[c].std(ddof=0) > 0]
        x_raw = x_raw[keep_cols]
        if x_raw.empty or len(g) <= x_raw.shape[1] + 1:
            rows.append({"strategy": strategy, "alpha": np.nan, "alpha_t": np.nan, "factor_beta_norm": np.nan, "r_squared": np.nan, "n_obs": int(len(g)), "status": "insufficient_df"})
            continue
        x = sm.add_constant(x_raw, has_constant="add")
        model = sm.OLS(g["strategy_return"], x).fit()
        betas = model.params.drop("const", errors="ignore")
        rows.append(
            {
                "strategy": strategy,
                "alpha": float(model.params.get("const", np.nan)),
                "alpha_t": float(model.tvalues.get("const", np.nan)),
                "factor_beta_norm": float(np.sqrt(np.sum(np.square(betas)))) if len(betas) else np.nan,
                "r_squared": float(model.rsquared),
                "n_obs": int(model.nobs),
                "status": "ok",
            }
        )
    return pd.DataFrame(rows)


def run_interaction_regression(panel: pd.DataFrame) -> pd.DataFrame:
    design = panel[["future_residual_return", "past_residual_momentum", "coherence_z"]].copy().dropna()
    design["interaction"] = design["past_residual_momentum"] * design["coherence_z"]
    required = {"future_residual_return", "past_residual_momentum", "coherence_z", "interaction"}
    assert required.issubset(design.columns)
    if len(design) <= 4:
        return pd.DataFrame([{"term": "interaction", "coef": np.nan, "t_stat": np.nan, "n_obs": int(len(design)), "status": "too_few_obs"}])
    x = sm.add_constant(design[["past_residual_momentum", "coherence_z", "interaction"]], has_constant="add")
    if np.linalg.matrix_rank(x.to_numpy()) < x.shape[1]:
        return pd.DataFrame([{"term": "interaction", "coef": np.nan, "t_stat": np.nan, "n_obs": int(len(design)), "status": "rank_deficient"}])
    model = sm.OLS(design["future_residual_return"], x).fit()
    return pd.DataFrame(
        [
            {"term": term, "coef": float(model.params[term]), "t_stat": float(model.tvalues[term]), "n_obs": int(model.nobs), "status": "ok"}
            for term in ["past_residual_momentum", "coherence_z", "interaction"]
        ]
    )


def pca_stats_for_window(window: pd.DataFrame) -> dict[str, float]:
    clean = window.dropna(axis=1, thresh=max(3, int(np.ceil(len(window) * cfg.min_obs_ratio))))
    if clean.shape[1] < 2:
        return {"first_eigenvalue": np.nan, "first_pc_share": np.nan, "apwc": np.nan, "n_tickers": clean.shape[1]}
    corr = clean.corr().fillna(0.0)
    eigvals = np.linalg.eigvalsh(corr.to_numpy(dtype=float))[::-1]
    eigvals = np.maximum(eigvals, 0.0)
    total = eigvals.sum()
    return {
        "first_eigenvalue": float(eigvals[0]),
        "first_pc_share": float(eigvals[0] / total) if total > 0 else np.nan,
        "apwc": average_pairwise_corr(clean),
        "n_tickers": int(clean.shape[1]),
    }


def run_residual_pca_diagnostics(panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    for _, row in panel.iterrows():
        constituents_now = latest_constituents(theme_constituents, row["theme_id"], row["analysis_date"], public_only=True)
        tickers = [t for t in constituents_now["ticker"] if t in residuals_wide.columns]
        window = residuals_wide.loc[(residuals_wide.index >= row["past_start"]) & (residuals_wide.index <= row["past_end"]), tickers]
        stats = pca_stats_for_window(window)
        stats.update({"theme_id": row["theme_id"], "analysis_date": row["analysis_date"], "coherence_top50": bool(row["coherence_top50"])})
        rows.append(stats)
    detail = pd.DataFrame(rows)
    summary = detail.groupby("coherence_top50").agg(
        rows=("theme_id", "size"),
        mean_first_eigenvalue=("first_eigenvalue", "mean"),
        mean_first_pc_share=("first_pc_share", "mean"),
        mean_apwc=("apwc", "mean"),
    ).reset_index()
    return detail, summary


def run_placebo_matched_baskets(panel: pd.DataFrame, n_placebo_per_row: int = 1, seed: int = 31415) -> tuple[pd.DataFrame, pd.DataFrame]:
    local_rng = np.random.default_rng(seed)
    all_theme_tickers = set(theme_constituents["ticker"])
    eligible = np.array(sorted(set(returns_wide.columns) - all_theme_tickers), dtype=object)
    if len(eligible) == 0:
        eligible = np.array(sorted(returns_wide.columns), dtype=object)
    rows = []
    for _, row in panel.iterrows():
        n = min(int(row["n_members"]), len(eligible))
        for k in range(n_placebo_per_row):
            sample = list(local_rng.choice(eligible, size=n, replace=False))
            past_dates = trading_dates_between(residuals_wide.index, row["past_start"], row["past_end"])
            future_dates = trading_dates_between(residuals_wide.index, row["future_start"], row["future_end"])
            placebo_past = residuals_wide.loc[past_dates, sample].mean(axis=1).sum()
            placebo_future = residuals_wide.loc[future_dates, sample].mean(axis=1).sum()
            placebo_apwc = average_pairwise_corr(residuals_wide.loc[past_dates, sample])
            rows.append(
                {
                    "theme_id": row["theme_id"],
                    "analysis_date": row["analysis_date"],
                    "placebo_id": k,
                    "n_tickers": n,
                    "placebo_apwc": placebo_apwc,
                    "placebo_past_residual_momentum": float(placebo_past),
                    "placebo_future_residual_return": float(placebo_future),
                    "actual_apwc": row["apwc"],
                    "actual_future_residual_return": row["future_residual_return"],
                }
            )
    detail = pd.DataFrame(rows)
    summary = pd.DataFrame(
        [
            {
                "placebo_rows": int(len(detail)),
                "placebo_mean_apwc": float(detail["placebo_apwc"].mean()) if len(detail) else np.nan,
                "actual_mean_apwc": float(panel["apwc"].mean()) if len(panel) else np.nan,
                "placebo_mean_future_residual_return": float(detail["placebo_future_residual_return"].mean()) if len(detail) else np.nan,
                "actual_mean_future_residual_return": float(panel["future_residual_return"].mean()) if len(panel) else np.nan,
            }
        ]
    )
    return detail, summary

coherence_momentum_panel = build_coherence_momentum_panel(tradable_results)
coherence_momentum_panel_available = len(coherence_momentum_panel) > 0
momentum_comparison, strategy_return_panel = compare_momentum_by_coherence(coherence_momentum_panel)
factor_decomposition_table = run_factor_decomposition(strategy_return_panel)
interaction_regression_table = run_interaction_regression(coherence_momentum_panel)
pca_detail, pca_group_summary = run_residual_pca_diagnostics(coherence_momentum_panel)
placebo_detail, placebo_summary = run_placebo_matched_baskets(coherence_momentum_panel, n_placebo_per_row=1, seed=cfg.seed + 555)

coherence_top50_momentum_mean_return = float(momentum_comparison.loc[momentum_comparison["strategy"].eq("coherence_top50_momentum"), "future_total_return_mean"].iloc[0])
all_momentum_mean_return = float(momentum_comparison.loc[momentum_comparison["strategy"].eq("all_momentum"), "future_total_return_mean"].iloc[0])
interaction_term_t_stat = float(interaction_regression_table.loc[interaction_regression_table["term"].eq("interaction"), "t_stat"].iloc[0])
coh_alpha = factor_decomposition_table.loc[factor_decomposition_table["strategy"].eq("coherence_top50_momentum"), "alpha_t"]
factor_decomposition_alpha_t = float(coh_alpha.iloc[0]) if len(coh_alpha) else np.nan
pca_high = pca_group_summary.loc[pca_group_summary["coherence_top50"].eq(True), "mean_first_eigenvalue"]
pca_low = pca_group_summary.loc[pca_group_summary["coherence_top50"].eq(False), "mean_first_eigenvalue"]
pca_high_vs_low_first_eigen_ratio = float(pca_high.iloc[0] / pca_low.iloc[0]) if len(pca_high) and len(pca_low) and pca_low.iloc[0] != 0 else np.nan
coherence_momentum_interpretation_ready = not USING_SYNTHETIC_DATA

assert coherence_momentum_panel.groupby("analysis_date")["coherence_top50"].sum().min() >= 1
assert (coherence_momentum_panel["past_end"] <= coherence_momentum_panel["analysis_date"]).all()
assert (coherence_momentum_panel["future_start"] > coherence_momentum_panel["analysis_date"]).all()
assert {"past_residual_momentum", "coherence_z", "future_residual_return"}.issubset(coherence_momentum_panel.columns)

print("coherence_momentum_interpretation_ready =", coherence_momentum_interpretation_ready)
print("synthetic note: values are smoke-test diagnostics, not empirical findings." if USING_SYNTHETIC_DATA else "real-data mode: diagnostics can be interpreted after data QA.")
display(coherence_momentum_panel[["theme_id", "analysis_date", "apwc", "z_score", "coherence_top50", "past_total_momentum", "future_total_return", "future_residual_return"]])
display(momentum_comparison)
display(strategy_return_panel)
display(factor_decomposition_table)
display(interaction_regression_table)
display(pca_group_summary)
display(placebo_summary)

## 12. 月次リバランス時点の整合例

月末営業日 `signal_date` で計算したシグナルは、翌営業日 `apply_from` から使う。

In [ ]:
def next_trading_day(date: pd.Timestamp, trading_dates: pd.Index) -> pd.Timestamp | pd.NaT:
    pos = int(np.searchsorted(trading_dates.values, np.datetime64(date), side="right"))
    if pos >= len(trading_dates):
        return pd.NaT
    return pd.Timestamp(trading_dates[pos])

month_ends = pd.Series(residuals_wide.index, index=residuals_wide.index).groupby(residuals_wide.index.to_period("M")).max()
rebalance_calendar = pd.DataFrame({"signal_date": month_ends.values})
rebalance_calendar["apply_from"] = rebalance_calendar["signal_date"].map(lambda x: next_trading_day(pd.Timestamp(x), residuals_wide.index))
rebalance_calendar = rebalance_calendar.dropna().tail(6)
display(rebalance_calendar)
assert (rebalance_calendar["apply_from"] > rebalance_calendar["signal_date"]).all()

## 13. 文献対応表

文献要件とNotebook実装の対応を明示する。

In [ ]:
alignment_table = pd.DataFrame(
    [
        ("Factor model: r = Xb + u", "Defining Themes / Eq. (1)", "Rank check付き日次OLSでfactor returnとresidualを推定", "Implemented", "定数market exposureはdropしてdiagnosticsへ記録"),
        ("Use of daily excess returns", "Empirical Results", "stock_excess_returnsを優先。total_return fallbackはpaper_replication_ready=False", "Implemented", PAPER_REPLICATION_REASON),
        ("Theme as transient residual correlation", "Defining Themes / Eq. (3)", "APWC z-scoreでcoherent themeを判定", "Implemented", "合成データ結果はsmoke test"),
        ("APWC definition", "Testing for Coherent Themes / Eq. (4)", "average_pairwise_corrで相関行列上三角平均", "Implemented", "Notebook内unit testsあり"),
        ("Bootstrap z-score definition", "Statistical Tests / Eq. (8)", "observed APWC と null APWC の平均・標準偏差からz-score", "Implemented", "null_stdが不足する場合はNaN"),
        ("Mosaic + bootstrap tiling", "Statistical Tests", "mosaic_bootstrap_apwc_paper_likeで10日batch x stock groupのtile別残差化", "Paper-like", "完全なSpector et al.実装ではないためwarningsに明記"),
        ("Release pre/post 60 calendar day analysis", "Exhibit 7", "compute_release_window_zscoresでpre/postを60 calendar daysに固定", "Implemented", "preはis_ex_post=True"),
        ("z >= 2 coherent threshold", "Exhibit 7 discussion", "cfg.z_threshold=2.0", "Implemented", "pre/post/tradableで使用"),
        ("Risk underestimation from nonzero residual correlations", "Themes as a Source of Risk and Return / Exhibit 8", "sample_diag対full covarianceの残差リスク倍率", "Implemented", "現状はin-sample sample_diag"),
        ("Future residual return regression by coherent vs non-coherent groups", "Exhibits 9/10", "persistence_regressionで群別回帰", "Implemented with caveat", "合成データ時はinterpretation_ready=False"),
        ("Random basket calibration", "Statistical Tests / random basket discussion", "random_basket_calibrationで負例z-scoreを集計", "Implemented", "合成データではテーマ銘柄を除外"),
    ],
    columns=["Literature requirement", "Paper location / section", "Notebook implementation", "Status", "Notes"],
)
display(alignment_table)

## 14. Validation report

最後に実行状態を集約する。実データ未配置時は `real_data_validation = not_run_synthetic_mode` であり、数値結果は研究結果ではない。

In [ ]:
mosaic_method_used = "mosaic_bootstrap_paper_like"
real_data_validation = "not_run_synthetic_mode" if USING_SYNTHETIC_DATA else "schema_loaded_and_executed"
validation_report = {
    "using_synthetic_data": USING_SYNTHETIC_DATA,
    "real_data_validation": real_data_validation,
    "synthetic_dgp": cfg.synthetic_dgp if USING_SYNTHETIC_DATA else None,
    "return_type": RETURN_TYPE,
    "return_construction": RETURN_CONSTRUCTION,
    "paper_replication_ready": PAPER_REPLICATION_READY,
    "paper_replication_reason": PAPER_REPLICATION_REASON,
    "n_themes": int(theme_constituents["theme_id"].nunique()),
    "n_tickers": int(stock_returns["ticker"].nunique()),
    "n_return_dates": int(stock_returns["date"].nunique()),
    "factor_model_rank_issues": factor_model_rank_issues,
    "max_abs_daily_residual_mean": max_abs_residual_mean,
    "max_abs_factor_residual_inner_product": max_abs_factor_residual_inner_product,
    "apwc_unit_tests_passed": bool(apwc_unit_tests_passed),
    "release_window_mode_available": bool(release_window_mode_available),
    "tradable_rolling_timing_ok": bool(tradable_rolling_timing_ok),
    "mosaic_method_used": mosaic_method_used,
    "simplified_bootstrap_used_only_for_demo": bool(simplified_bootstrap_used_only_for_demo),
    "random_basket_calibration_mean_z": random_basket_calibration_mean_z,
    "random_basket_calibration_share_abs_z_gt_2": random_basket_calibration_share_abs_z_gt_2,
    "persistence_interpretation_ready": bool(persistence_interpretation_ready),
    "coherence_momentum_panel_available": bool(coherence_momentum_panel_available),
    "coherence_top50_momentum_mean_return": coherence_top50_momentum_mean_return,
    "all_momentum_mean_return": all_momentum_mean_return,
    "interaction_term_t_stat": interaction_term_t_stat,
    "factor_decomposition_alpha_t": factor_decomposition_alpha_t,
    "pca_high_vs_low_first_eigen_ratio": pca_high_vs_low_first_eigen_ratio,
    "coherence_momentum_interpretation_ready": bool(coherence_momentum_interpretation_ready),
    "notebook_scope": "single_notebook_no_external_py",
}
display(pd.Series(validation_report, name="validation_report"))

summary_tables = {
    "release_window_summary": release_window_summary,
    "risk_group_summary": risk_group_summary,
    "persistence_table": persistence_table,
    "random_basket_summary": pd.Series(random_summary).to_frame("value"),
    "coherence_momentum_comparison": momentum_comparison,
    "coherence_factor_decomposition": factor_decomposition_table,
    "coherence_interaction_regression": interaction_regression_table,
    "coherence_pca_group_summary": pca_group_summary,
    "coherence_placebo_summary": placebo_summary,
}
for name, table in summary_tables.items():
    print(f"\n{name}")
    display(table)